<a href="https://colab.research.google.com/github/AHNK-05/Python_trading_system/blob/main/algo_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import yfinance as yf
import numpy as np
import pandas as pd
import math
import requests

import os
import io

In [12]:

# Official NSE Indices constituent files (tried in order).
CONSTITUENT_URLS = [
    "https://www.niftyindices.com/IndexConstituent/ind_nifty50list.csv",
    "https://archives.nseindia.com/content/indices/ind_nifty50list.csv",
    "https://nsearchives.nseindia.com/content/indices/ind_nifty50list.csv",
]

In [13]:

# NSE blocks requests that don't look like they come from a browser.
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    ),
    "Accept": "text/csv,*/*",
}

SYMBOLS_CACHE = "nifty50_symbols.csv"


In [14]:

def get_nifty50_symbols(use_cache_on_failure: bool = True) -> list[str]:
    """Return the current Nifty 50 NSE symbols.

    Downloads the official list from NSE Indices and saves a local copy.
    If every download fails, falls back to the last saved copy.
    """
    last_error = None
    for url in CONSTITUENT_URLS:
        try:
            response = requests.get(url, headers=HEADERS, timeout=20)
            response.raise_for_status()
            df = pd.read_csv(io.StringIO(response.text))
            df.columns = [c.strip() for c in df.columns]
            symbols = df["Symbol"].str.strip().tolist()
            if len(symbols) != 50:
                print(f"Warning: expected 50 symbols, got {len(symbols)}")
            df.to_csv(SYMBOLS_CACHE, index=False)  # keep a copy for offline use
            return symbols
        except Exception as exc:  # try the next URL
            last_error = exc

    if use_cache_on_failure and os.path.exists(SYMBOLS_CACHE):
        print(f"Could not download the list ({last_error}). Using cached {SYMBOLS_CACHE}.")
        return pd.read_csv(SYMBOLS_CACHE)["Symbol"].str.strip().tolist()

    raise RuntimeError(f"Could not fetch the Nifty 50 list: {last_error}")

In [15]:
symbols = get_nifty50_symbols()
print(len(symbols))   # should be 50
print(symbols)

50
['ADANIENT', 'ADANIPORTS', 'APOLLOHOSP', 'ASIANPAINT', 'AXISBANK', 'BSE', 'BAJAJ-AUTO', 'BAJFINANCE', 'BAJAJFINSV', 'BEL', 'BHARTIARTL', 'CIPLA', 'COALINDIA', 'DRREDDY', 'EICHERMOT', 'ETERNAL', 'GRASIM', 'HCLTECH', 'HDFCBANK', 'HDFCLIFE', 'HINDALCO', 'HINDUNILVR', 'ICICIBANK', 'ITC', 'INFY', 'INDIGO', 'JSWSTEEL', 'JIOFIN', 'KOTAKBANK', 'LT', 'M&M', 'MARUTI', 'MAXHEALTH', 'NTPC', 'NESTLEIND', 'ONGC', 'POWERGRID', 'RELIANCE', 'SBILIFE', 'SHRIRAMFIN', 'SBIN', 'SUNPHARMA', 'TCS', 'TATACONSUM', 'TMPV', 'TATASTEEL', 'TECHM', 'TITAN', 'TRENT', 'ULTRACEMCO']


In [21]:
stocks = pd.read_csv('nifty50_symbols.csv')
stocks = stocks['Symbol']